# 🧠 Brain Tumor Detection: Multi-Model Benchmark & Comparative Analysis

This notebook presents an in-depth comparative study of multiple deep learning architectures for **Brain Tumor MRI Classification**:

1. **Custom Deep CNN**: 4-stage convolutional neural network built from scratch with batch normalization, leaky activations, and dropout.
2. **ResNet-18 (Transfer Learning)**: Residual learning framework leveraging skip connections and pre-trained ImageNet representations.
3. **MobileNetV3 (Hardware-Efficient CNN)**: High-speed, parameter-efficient neural network optimized for low latency.
4. **Vision Transformer (ViT-B/16)**: Self-attention transformer processing non-overlapping image patches.

---

### 🎯 Evaluation & Comparison Objectives
- **Error Metrics**: Cross-Entropy (Log Loss), Top-1 Error Rate ($1 - \text{Accuracy}$), Misclassification Counts, Class-specific Miss Rates.
- **Accuracy Metrics**: Test Accuracy (%), Balanced Accuracy (%), Precision, Recall, Macro & Weighted F1-Scores.
- **Discrimination & Calibration**: Multi-class One-vs-Rest ROC-AUC curves.
- **Error Analysis**: Confusion matrices and identification of common failure modes (e.g., Glioma vs. Meningioma margin confusion).
- **Efficiency**: Model parameter size vs. inference latency trade-offs.

In [ ]:
# Cell 1: Environment Setup and Imports
import os
import sys
from pathlib import Path

# Ensure project root is in python path
ROOT_DIR = Path(".").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from src.config import CLASS_NAMES, CLASS_METADATA, DATA_DIR, TEST_DIR, TRAIN_DIR, get_device
from src.data import get_image_folder, get_data_loaders
from src.models import MODEL_REGISTRY, build_model, count_parameters
from src.metrics import evaluate_model_on_loader

device = get_device()
print(f"Active Device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"Target Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

## 1. Dataset Exploration & Class Distribution
Let's inspect the distribution of samples across the **Training** and **Testing** splits.

In [ ]:
# Cell 2: Explore Dataset Splits and Image Counts
splits = {'Training': TRAIN_DIR, 'Testing': TEST_DIR}
counts = []

for split_name, split_path in splits.items():
    for cls in CLASS_NAMES:
        folder = split_path / cls
        count = len(list(folder.glob('*.jpg'))) if folder.exists() else 0
        counts.append({'Split': split_name, 'Class': cls.capitalize(), 'Images': count})

df_counts = pd.DataFrame(counts)
print(df_counts.pivot(index='Class', columns='Split', values='Images'))

# Plot Class Distribution
plt.figure(figsize=(9, 4.5))
sns.barplot(data=df_counts, x='Class', y='Images', hue='Split', palette=['#2a9d8f', '#e76f51'])
plt.title("Dataset Sample Distribution per Class & Split")
plt.ylabel("Number of Scans")
plt.show()

## 2. Sample Brain MRI Visualizations
Visualize representative scans from each class.

In [ ]:
# Cell 3: Display Sample MRI Scans from each class
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, cls in enumerate(CLASS_NAMES):
    folder = TRAIN_DIR / cls
    sample_file = list(folder.glob('*.jpg'))[0]
    img = Image.open(sample_file).convert('RGB')
    
    axes[idx].imshow(img)
    axes[idx].set_title(f"{cls.capitalize()}", fontsize=13, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle("Sample MRI Scans Across Tumor Categories", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 3. Model Architecture Specifications & Complexity Comparison
Compare parameter footprints and structural properties of the 4 candidate models.

In [ ]:
# Cell 4: Parameter & Architecture Comparison
arch_summary = []
for key, spec in MODEL_REGISTRY.items():
    model = build_model(key, num_classes=len(CLASS_NAMES), pretrained=False)
    params = count_parameters(model)
    arch_summary.append({
        'Model': spec['display_name'],
        'Type': spec['description'].split(' ')[0],
        'Total Parameters': f"{params['total']:,}",
        'Trainable Parameters': f"{params['trainable']:,}",
        'Size on Disk (~MB)': f"{round(params['total'] * 4 / (1024**2), 2)} MB"
    })

pd.DataFrame(arch_summary)

## 4. Multi-Model Benchmark & Error/Accuracy Comparison
Evaluate test set predictions, calculate comprehensive loss, error, accuracy, and F1 scores.

In [ ]:
# Cell 5: Run Evaluation on Test Set
from evaluate import evaluate_all_models

results = evaluate_all_models(test_dir=TEST_DIR, batch_size=32, device=device)
df_results = pd.read_csv('results/model_comparison_metrics.csv')
display(df_results)

## 5. Comparative Visualizations
### 5.1 Test Accuracy vs. Error Rate
Highlighting performance and error trade-offs.

In [ ]:
# Cell 6: Accuracy vs Error Rate Visualization
from compare import plot_accuracy_and_error, plot_f1_and_auc, plot_per_class_f1, plot_confusion_matrices

models = df_results['Model']
acc = df_results['Test Accuracy (%)']
err = df_results['Error Rate (%)']

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(x - width/2, acc, width, label='Test Accuracy (%)', color='#2a9d8f')
ax.bar(x + width/2, err, width, label='Error Rate (%)', color='#e76f51')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('Percentage (%)')
ax.set_title('Test Accuracy vs. Top-1 Error Rate Across Models')
ax.legend()
plt.tight_layout()
plt.show()

### 5.2 Confusion Matrices Comparison
Side-by-side confusion matrices show how well each model differentiates adjacent pathologies.

In [ ]:
# Cell 7: Side-by-Side Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (key, data) in enumerate(results.items()):
    cm = np.array(data['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[idx].set_title(f"{data['display_name']} (Acc: {data['test_accuracy_pct']}%)")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Ground Truth')

plt.tight_layout()
plt.show()

### 5.3 Per-Class F1 Breakdown
Examine per-class sensitivities for Glioma, Meningioma, Pituitary, and Normal scans.

In [ ]:
# Cell 8: Per-Class F1 Heatmap
class_f1_matrix = []
for key, data in results.items():
    class_f1_matrix.append([
        data['glioma_f1'],
        data['meningioma_f1'],
        data['notumor_f1'],
        data['pituitary_f1']
    ])

plt.figure(figsize=(9, 4))
sns.heatmap(
    class_f1_matrix,
    annot=True,
    fmt='.3f',
    cmap='YlGnBu',
    xticklabels=[c.capitalize() for c in CLASS_NAMES],
    yticklabels=[data['display_name'] for data in results.values()]
)
plt.title('Tumor Pathology-Specific F1-Score Matrix')
plt.show()

## 6. Key Conclusions & Findings
- **Accuracy & Error Comparison**: How pre-trained feature extractors (ResNet-18 / ViT) compare against custom CNN.
- **Pathology-Specific Challenges**: Gliomas tend to exhibit higher intra-class variation than meningiomas and pituitary adenomas.
- **Efficiency / Latency**: MobileNetV3 achieves near-transfer learning accuracy while using a fraction of the computational requirements.